<a href="https://colab.research.google.com/github/purnabandekar/TNS/blob/main/Clustering_KMeans_Practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clustering — K-Means and Hierarchical

### Python Course · Day 14 · Colab Notebook

---

Everything we have built so far came with an **answer column** — pass or fail, cancer or
healthy, a house price. The model's job was to copy those answers.

**Today there is no answer column.** We have 200 customers and nothing else, and we want the
computer to work out for itself who belongs with whom.

---

**How to use this notebook**

1. Run every cell in order, top to bottom. Click a cell and press **Shift + Enter**.
2. **Every line of code has a comment** explaining what it does.
3. Most sections end with a **YOUR TURN** box — try it before moving on.
4. **Section 12 is a set of exercises with worked answers.**
5. Nothing to install, nothing to upload.

**Save your own copy first:** File → Save a copy in Drive.

---

### What we are covering

| Section | Topic |
|---|---|
| 1 | The customers |
| 2 | K-means in three lines |
| 3 | Reading the groups |
| 4 | Doing one round by hand |
| 5 | Choosing k — the elbow |
| 6 | Choosing k — the silhouette |
| 7 | Scale changes everything |
| 8 | Hierarchical clustering |
| 9 | The dendrogram |
| 10 | Cutting the tree |
| 11 | Where k-means struggles |
| 12 | **Exercises**, with answers |

---
## Section 1 — The customers

We will make the data ourselves so that everybody's numbers match and nothing has to be
downloaded.

Each customer has two facts:

- **annual_income** — their yearly income in thousands,
- **spending_score** — how much they spend at this shop, on a scale of 1 to 100.

We are secretly building them out of five groups. **We will not tell the computer that** —
that is the whole point. We keep the secret so that at the very end we can check whether it
found them.

In [ ]:
# import numpy for the random numbers
import numpy as np
# import pandas to hold the data as a table
import pandas as pd
# import matplotlib for the pictures
import matplotlib.pyplot as plt

# a random number generator with a fixed seed, so your customers match mine
rng = np.random.default_rng(7)

# the middle point of each of our five secret groups: (income, spending)
group_centres = [(25, 20), (25, 78), (55, 50), (85, 18), (85, 80)]
# how many customers we want in each of those groups
group_sizes = [38, 36, 62, 32, 32]

In [ ]:
# an empty list to collect every customer's income
incomes = []
# an empty list to collect every customer's spending score
scores = []
# an empty list remembering which secret group each customer came from
secret_group = []

# build one group at a time, five times in total
for i in range(5):
    # pull out the middle point of this group
    middle_income = group_centres[i][0]
    # and the middle spending score of this group
    middle_score = group_centres[i][1]
    # scatter this group's customers around that middle point
    # normal() means most land near the middle and a few land further out
    these_incomes = rng.normal(middle_income, 6, group_sizes[i])
    # do the same for their spending scores
    these_scores = rng.normal(middle_score, 7, group_sizes[i])
    # add each customer of this group onto the end of our three lists
    for j in range(group_sizes[i]):
        # round to 1 decimal place so the table is readable
        incomes.append(round(float(these_incomes[j]), 1))
        # store this customer's spending score too
        scores.append(round(float(these_scores[j]), 1))
        # remember which group they really came from - this is our secret
        secret_group.append(i)

In [ ]:
# put the three lists into one table
customers = pd.DataFrame({"annual_income": incomes,
                          "spending_score": scores,
                          "secret_group": secret_group})

# shuffle the rows so the groups are not sitting in neat blocks
# frac=1 means "give me back all of the rows", just in a new order
customers = customers.sample(frac=1, random_state=1).reset_index(drop=True)

# show the first 8 customers
print(customers.head(8))
# and how many we have in total
print("total customers:", len(customers))

Now we split the table in two: the part the algorithm is allowed to see, and the secret.

In [ ]:
# X is what we hand to the clustering algorithm - just the two facts, no answer
X = customers[["annual_income", "spending_score"]]

# the secret group is kept aside, and we will not touch it until Section 12
secret = customers["secret_group"]

# show that X really has no answer column in it
print(X.head(3))

In [ ]:
# set the size of the picture
plt.figure(figsize=(6, 4.5))
# draw one grey dot per customer - all the same colour, because we have no groups yet
plt.scatter(X["annual_income"], X["spending_score"], color="grey", s=25)
# label the bottom axis
plt.xlabel("annual income (thousands)")
# label the side axis
plt.ylabel("spending score (1-100)")
# give the picture a title
plt.title("200 customers, before any clustering")
# show the picture
plt.show()

Your eyes can already pick out the groups. **The computer cannot see** — it only has two
columns of numbers, and it needs a method.

> ### YOUR TURN
> Before running anything else, count the groups you can see in that picture. Write the
> number down. We will find out later whether the computer agrees with you.

---
## Section 2 — K-means in three lines

`n_clusters=5` is us telling it how many groups to look for. `fit_predict` does the work and
hands back one group number per customer.

Two settings worth knowing:

- **`n_init=10`** — where the centres start is chosen partly at random, so it runs the whole
  thing 10 times and keeps the best attempt. (Sklearn does not scatter them completely at
  random; it deliberately picks starting points that are spread out.)
- **`random_state=0`** — fixes those random starts, so your answer matches mine.

In [ ]:
# bring in k-means
from sklearn.cluster import KMeans

# create the model, asking for 5 groups
kmeans = KMeans(n_clusters=5, n_init=10, random_state=0)

# fit it to the data and get back one group number per customer
labels = kmeans.fit_predict(X)

# print the group number of the first 15 customers
print("first 15 labels:", labels[:15])

Those numbers are **names, not scores**. Group 3 is not better than group 1, and not three
times anything. They are just five different labels.

In [ ]:
# set the picture size
plt.figure(figsize=(6.5, 4.8))
# colour every dot by its group number, using a colour scheme called tab10
plt.scatter(X["annual_income"], X["spending_score"], c=labels, cmap="tab10", s=28)
# draw a big black X at each group's centre
# cluster_centers_ holds one row per group: [income, spending]
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            color="black", marker="X", s=220)
# label the bottom axis
plt.xlabel("annual income (thousands)")
# label the side axis
plt.ylabel("spending score (1-100)")
# title the picture
plt.title("k-means with k = 5")
# show it
plt.show()

> ### YOUR TURN
> Does the computer's answer match the number of groups you wrote down in Section 1?

---
## Section 3 — Reading the groups

A picture is nice, but the useful output is a description of each group that you could hand
to somebody in marketing.

In [ ]:
# put the group numbers back onto the customer table as a new column
customers["cluster"] = labels

# group the table by cluster and take the average of each column
# round(1) keeps the numbers short enough to read
summary = customers.groupby("cluster")[["annual_income", "spending_score"]].mean().round(1)

# print the average customer in each group
print(summary)

In [ ]:
# count how many customers landed in each group
print(customers["cluster"].value_counts().sort_index())

### Turning that into English

Read the two numbers on each row and give the group a name. For example, a group with a low
income and a high spending score is *young spenders*; a high income with a low score is
*rich but careful*.

**Those names are yours, not the computer's.** All it ever said was 0, 1, 2, 3 and 4. Giving
the groups meaning is the part of clustering that only a person can do.

> ### YOUR TURN
> Write a one-line name for each of the five rows in that summary table. Which group would
> you send a discount voucher to, and why?

---
## Section 4 — Doing one round by hand

k-means is only two steps, repeated. Let us do those two steps ourselves, so nothing is
hidden inside the library.

1. **Assign** — every customer joins whichever centre is nearest.
2. **Move** — every centre slides to the middle of the customers who joined it.

In [ ]:
# turn the table into a plain grid of numbers, which is easier to do maths on
points = X.to_numpy()

# start with five made-up centres, deliberately in silly positions
centres = np.array([[15.0, 30.0], [30.0, 40.0], [45.0, 60.0], [60.0, 35.0], [70.0, 55.0]])

# print a heading
print("starting centres:")
# print the five starting positions
print(centres)

In [ ]:
# STEP 1 - work out which centre each customer is nearest to

# a list to collect one group number per customer
my_labels = []

# look at every customer, one at a time
for i in range(len(points)):
    # start by assuming the first centre is the nearest
    best_centre = 0
    # and work out how far away it is
    # the distance between two points is the square root of
    # (difference in x) squared plus (difference in y) squared
    best_distance = np.sqrt((points[i][0] - centres[0][0]) ** 2 +
                            (points[i][1] - centres[0][1]) ** 2)
    # now check the other four centres to see if any is closer
    for c in range(1, 5):
        # the distance from this customer to centre number c
        d = np.sqrt((points[i][0] - centres[c][0]) ** 2 +
                    (points[i][1] - centres[c][1]) ** 2)
        # if this one is closer than the best so far, it becomes the new best
        if d < best_distance:
            # remember the new shortest distance
            best_distance = d
            # and remember which centre it belongs to
            best_centre = c
    # record the winning centre for this customer
    my_labels.append(best_centre)

# turn the list into a numpy array so we can compare it with numbers easily
my_labels = np.array(my_labels)
# count how many customers joined each centre
print("customers per centre:", np.bincount(my_labels))

In [ ]:
# STEP 2 - move each centre to the middle of the customers who joined it

# make a copy so we can compare the before and after
new_centres = centres.copy()

# do this for each of the five centres
for c in range(5):
    # pick out only the customers who joined centre c
    mine = points[my_labels == c]
    # if anybody joined it, move it to the average of those customers
    if len(mine) > 0:
        # mean(axis=0) takes the average down each column: income, then spending
        new_centres[c] = mine.mean(axis=0)

# show where each centre started and where it moved to
for c in range(5):
    # print the centre number, its old position and its new position
    print("centre", c,
          "from", np.round(centres[c], 1),
          "to", np.round(new_centres[c], 1))

That is one round. Real k-means simply does those two steps again and again until the
centres stop moving — usually after a handful of rounds.

> ### YOUR TURN
> As written, running the two cells again changes nothing: Step 1 always measures from
> `centres`, which we never update.
>
> Add the line `centres = new_centres.copy()` to the very end of the Step 2 cell, then run
> Step 1 and Step 2 a few more times and watch the centres settle.
>
> Four of the five calm down quickly. **One of them jumps further on the second round than
> it did on the first** — find it, and work out why. (Look at how many customers joined each
> centre the first time.)
>
> Nothing later in the notebook uses `centres`, so this edit is safe to leave in.

---
## Section 5 — Choosing k: the elbow

We told it 5. But how would we know, on data we had never seen?

**Inertia** is the number k-means is trying to make small: for every customer, the squared
distance to their own centre, all added up. Small inertia means tight groups.

In [ ]:
# the values of k we want to try
k_values = [1, 2, 3, 4, 5, 6, 7, 8]
# an empty list to collect the inertia at each k
inertias = []

# try each k in turn
for k in k_values:
    # build and fit a k-means with this many groups
    m = KMeans(n_clusters=k, n_init=10, random_state=0)
    # fit it to the customers
    m.fit(X)
    # inertia_ is the total spread inside the clusters
    inertias.append(m.inertia_)
    # print it as we go, rounded to whole numbers
    print("k =", k, "  inertia =", round(m.inertia_))

In [ ]:
# set the picture size
plt.figure(figsize=(6.5, 4.2))
# draw inertia against k as a line with round markers
plt.plot(k_values, inertias, marker="o")
# label the axes
plt.xlabel("k (number of clusters)")
# label the side axis
plt.ylabel("inertia")
# title the picture
plt.title("The elbow method")
# show it
plt.show()

### How to read it

Inertia **always** falls as k goes up — more groups means everybody is nearer to a centre.
So the smallest inertia is never the answer.

What we look for is the **bend**: the point after which extra groups stop buying you much.
Going from 4 to 5 removes a huge amount of inertia. Going from 5 to 6 barely moves it.

**That bend is at k = 5.**

> ### YOUR TURN
> If you had never seen the picture in Section 1, would you have been confident picking 5
> from this graph alone? Look at how sharp the corner is.

---
## Section 6 — Choosing k: the silhouette

A second opinion, and a more direct one. For each customer the **silhouette** compares how
close they are to their own group against how close they are to the nearest other group.

The score runs from **-1 to 1**:

- **near 1** — comfortably inside its own group,
- **near 0** — sitting on the boundary between two groups,
- **below 0** — probably in the wrong group.

`silhouette_score` averages that over everybody. **Highest wins — as long as there is a
clear peak.** (We will see in Section 12 why that qualifier matters.)

In [ ]:
# bring in the silhouette score
from sklearn.metrics import silhouette_score

# we start at 2 here, because a silhouette needs at least two groups to compare
for k in [2, 3, 4, 5, 6, 7, 8]:
    # fit a k-means with this many groups
    m = KMeans(n_clusters=k, n_init=10, random_state=0)
    # and get one group number per customer
    these_labels = m.fit_predict(X)
    # score how well separated those groups are
    score = silhouette_score(X, these_labels)
    # print the k and its score
    print("k =", k, "  silhouette =", round(score, 4))

The highest score is at **k = 5**, and it is not close. Two different methods, the same
answer — that is about as much reassurance as clustering ever gives you.

> ### YOUR TURN
> The silhouette at k = 2 is much lower than at k = 5. Look back at the picture in Section 1
> and say, in one sentence, what k-means would be forced to do to squeeze those five blobs
> into two groups.

---
## Section 7 — Scale changes everything

k-means measures **distance**. So a column with big numbers in it drowns out a column with
small numbers, no matter which one actually matters.

Watch. Nothing about our customers changes — we just write income in rupees instead of
thousands of rupees.

In [ ]:
# make a copy of the data so we do not spoil the original
X_rupees = X.copy()
# write income in rupees instead of thousands, so 25 becomes 25000
X_rupees["annual_income"] = X_rupees["annual_income"] * 1000

# print the first 3 rows so we can see what changed
print(X_rupees.head(3))

In [ ]:
# run exactly the same k-means on the rupee version
km_rupees = KMeans(n_clusters=5, n_init=10, random_state=0)
# fit it and collect the group numbers
labels_rupees = km_rupees.fit_predict(X_rupees)

# print a heading
print("group centres it found:")
# put the centres into a small table so both columns are easy to read
# round(0) drops the decimal places, and the column names remind us which is which
print(pd.DataFrame(km_rupees.cluster_centers_,
                   columns=["annual_income", "spending_score"]).round(0))

Look down the **second column** of those centres. Every single group has a spending score of
roughly 46 to 50 — the middle. The spending column has been ignored completely, and all it
has done is cut the income axis into five slices.

The fix is to **scale** the columns first, so both count the same amount.

In [ ]:
# bring in the scaler
from sklearn.preprocessing import StandardScaler

# StandardScaler rewrites each column so its average is 0 and its spread is exactly 1,
# which puts both columns on precisely the same footing
scaler = StandardScaler()
# apply it to the rupee data
X_scaled = scaler.fit_transform(X_rupees)

# run the same k-means again, this time on the scaled numbers
km_scaled = KMeans(n_clusters=5, n_init=10, random_state=0)
# fit it and collect the group numbers
labels_scaled = km_scaled.fit_predict(X_scaled)

In [ ]:
# draw the two answers side by side, 1 row of 2 pictures
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# left picture: the unscaled rupee answer, drawn on the original axes
axes[0].scatter(X["annual_income"], X["spending_score"], c=labels_rupees, cmap="tab10", s=20)
# title the left picture
axes[0].set_title("rupees, not scaled")

# right picture: the scaled answer, drawn on the same axes for a fair comparison
axes[1].scatter(X["annual_income"], X["spending_score"], c=labels_scaled, cmap="tab10", s=20)
# title the right picture
axes[1].set_title("rupees, scaled first")

# label both pictures
for ax in axes:
    # label the bottom axis of this picture
    ax.set_xlabel("annual income (thousands)")
    # label the side axis of this picture
    ax.set_ylabel("spending score")

# show them
plt.show()

Same data, same algorithm, same k. **Only the units changed** — and the left-hand answer is
useless.

Our original `X` happened to be safe, because income (10 to 97) and spending (6 to 96) were
already on a similar scale. That is luck, not good practice.

> ### YOUR TURN
> Which of these column pairs would give you trouble if you clustered them without scaling?
> Age in years and salary in rupees? Height in centimetres and weight in kilograms?

---
## Section 8 — Hierarchical clustering

A completely different idea:

1. Every customer starts as their own group — 200 groups.
2. Find the two closest groups and join them into one.
3. Repeat, until everything is in a single group — or until there are as many groups as you
   asked for.

`linkage="ward"` is the usual rule for deciding which pair is "closest": it joins whichever
pair keeps the resulting groups tightest.

**One honest note.** The idea does not need a k, but sklearn's `AgglomerativeClustering`
still makes you name one, so it knows when to stop — and it does not hand you the merge
history. To see the full tree we switch to a scipy function in the next section. **That** is
the version where you choose the number of groups afterwards.

In [ ]:
# bring in hierarchical clustering
from sklearn.cluster import AgglomerativeClustering

# create it, asking it to stop once it has 5 groups
hierarchical = AgglomerativeClustering(n_clusters=5, linkage="ward")

# fit it and get one group number per customer
labels_hc = hierarchical.fit_predict(X)

# print the first 15, next to what k-means said for the same customers
print("hierarchical:", labels_hc[:15])
# print what k-means said about the same 15 customers
print("k-means     :", labels[:15])

The two rows of numbers look different — but **the numbering is arbitrary**. What matters is
whether the same customers are grouped together, not what the groups are called.

A crosstab settles it. Each row is a k-means group, each column a hierarchical group.

In [ ]:
# count how many customers fall into each combination of the two labellings
print(pd.crosstab(labels, labels_hc))

Every row has all of its customers in exactly **one** column. The two methods put the same
200 people into the same 5 piles; they just numbered the piles differently.

> ### YOUR TURN
> Find the group of 62 in that table. Which k-means number and which hierarchical number
> does it have?

---
## Section 9 — The dendrogram

The real reward of hierarchical clustering is that you can **draw the whole merge history**.

We will draw 40 customers rather than 200, because 200 labels along the bottom is
unreadable.

In [ ]:
# bring in the two dendrogram tools from scipy
from scipy.cluster.hierarchy import dendrogram, linkage

# take a random 40 of our customers so the picture stays readable
sample = X.sample(n=40, random_state=3)

# linkage() does the merging and returns the full history
# "ward" is the same rule we used above
merge_history = linkage(sample, method="ward")

# set the picture size
plt.figure(figsize=(11, 5))
# draw the history as a tree
dendrogram(merge_history)
# label the axes
plt.xlabel("customers")
# label the side axis
plt.ylabel("how expensive that merge was")
# title the picture
plt.title("Dendrogram of 40 customers")
# show it
plt.show()

### How to read it

- Each **tick along the bottom** is one customer.
- Each **horizontal bar** is one merge: the two things underneath it were joined.
- The **height** of that bar is how **expensive** that merge was — how much extra spread it
  created inside the new group.

So low bars near the bottom are cheap merges between customers who were almost identical,
and a very tall bar is an expensive one, joining two groups that were nothing alike.

**A word of warning about that height.** With `ward` it is a cost, not a distance in income
and spending units. It grows with the *size* of the groups being joined as well as how far
apart they are — which is why the tallest bar here reads about 181, even though no two of
these 40 customers are more than about 105 apart. Compare heights with each other; do not
read them as gaps on the scatter plot.

> ### YOUR TURN
> Find the tallest bar. It is the very last merge. Which two big groups did it join, and why
> was joining them so expensive?

---
## Section 10 — Cutting the tree

Draw a horizontal line across the dendrogram and count how many vertical lines it crosses.
That count is your number of clusters.

**Cut in the biggest vertical gap** — the tall empty stretch where nothing merged. Merges
happening below the gap were cheap; merges above it were expensive.

In [ ]:
# the merge history has one row per merge, and column 2 is the height of that merge
merge_heights = merge_history[:, 2]

# print the last 8 merges, which are the tallest and the most interesting
print("heights of the last 8 merges:", np.round(merge_heights[-8:], 1))

In [ ]:
# draw the same tree again, with a line showing where we are cutting
plt.figure(figsize=(11, 5))
# color_threshold colours the branches below our cut, so the clusters stand out
dendrogram(merge_history, color_threshold=60)
# draw a red dashed line straight across at height 60
plt.axhline(60, color="red", linestyle="--")
# label the axes
plt.xlabel("customers")
# label the side axis
plt.ylabel("how expensive that merge was")
# title the picture
plt.title("Cutting at height 60")
# show it
plt.show()

In [ ]:
# bring in the tool that actually performs the cut
from scipy.cluster.hierarchy import fcluster

# cut the tree at height 60
# criterion="distance" means "60 is a height", not "60 clusters"
cut_labels = fcluster(merge_history, 60, criterion="distance")

# count how many different groups the cut produced
print("clusters after cutting at 60:", len(set(cut_labels)))
# and how big each one is
print("sizes:", np.bincount(cut_labels)[1:])

Look at the printed merge heights. There is a jump from about **32** straight up to about
**112** — nothing merged in between. Cutting anywhere in that empty stretch gives the same
answer, and it gives us **5** groups.

> ### YOUR TURN
> Change the `60` to `150` in the `fcluster` line and run it again. How many clusters do you
> get, and why did it drop?
>
> *(You should get 3 — only the two merges above 150 survive a cut that high. Put it back to
> 60 before you move on, so the printout matches the red line in the picture.)*

---
## Section 11 — Where k-means struggles

Every customer simply joins their nearest centre, so the groups k-means finds always come
out roughly **round**. When the real groups are long or curved, that is a problem.

Here are two crescent shapes that any person would separate instantly.

In [ ]:
# bring in a maker of crescent-shaped test data
from sklearn.datasets import make_moons

# build 300 points in two crescents, with a little noise to make it realistic
# the second thing it returns is the true group, which we throw away with _
moons, _ = make_moons(n_samples=300, noise=0.06, random_state=0)

# run k-means on it, asking for the 2 groups we can plainly see
km_moons = KMeans(n_clusters=2, n_init=10, random_state=0)
# fit it and collect the group numbers
moon_labels = km_moons.fit_predict(moons)

In [ ]:
# draw the two pictures side by side
figure, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# left: the raw shapes, all one colour
axes[0].scatter(moons[:, 0], moons[:, 1], color="grey", s=14)
# title the left picture
axes[0].set_title("what is really there")

# right: what k-means made of it
axes[1].scatter(moons[:, 0], moons[:, 1], c=moon_labels, cmap="tab10", s=14)
# title the right picture
axes[1].set_title("what k-means says")

# show them
plt.show()

k-means cut straight across both crescents, because a point at the tip of the top crescent
really is nearer to the bottom crescent's centre than to its own.

**It is not a bug.** k-means is doing exactly what it promises — nearest centre wins. It is
just the wrong promise for this shape.

> ### YOUR TURN
> Try `AgglomerativeClustering(n_clusters=2, linkage="single")` on `moons`. The `single`
> rule joins groups by their two closest points, which lets a cluster follow a chain around
> a curve. Does it do better?

---
## Section 12 — Exercises

Try each one before opening the answer underneath it.

### Exercise 1 — did it actually get it right?

Time to open the envelope. Compare the k-means groups against the secret groups we built the
data from in Section 1.

In [ ]:
# each row is a secret group, each column is the k-means group it ended up in
print(pd.crosstab(secret, labels))

**Answer.** Every secret group sits entirely in one k-means column, and every column holds
exactly one secret group. It got all 200 customers right.

Enjoy it, because it will not happen again. Our five groups were round, well separated and
about the right size — the ideal case for k-means. Real customer data overlaps, and there is
usually no secret answer to check against at all.

### Exercise 2 — the same shop, three groups

The marketing team says they only have the budget for **three** campaigns. Re-cluster with
k = 3 and describe what the computer does with your five groups.

In [ ]:
# cluster the same customers into 3 groups instead of 5
km3 = KMeans(n_clusters=3, n_init=10, random_state=0)
# fit it and collect the group numbers
labels3 = km3.fit_predict(X)

# make a copy of the table so we do not disturb the original
three = customers.copy()
# add the new group numbers as a column
three["cluster3"] = labels3

# show the average customer in each of the three groups
print(three.groupby("cluster3")[["annual_income", "spending_score"]].mean().round(1))
# and how many customers are in each
print(three["cluster3"].value_counts().sort_index())

**Answer.** It keeps the two **low-spending** groups apart — the careful savers on 22.6k and
the careful rich on 85.2k — and merges everybody else into one big group of 129, whose
average customer (54k, spending 64) is not really a customer at all: it is the middle of
three different groups averaged together.

It has not found anything new. It has **merged real groups**, because you told it to produce
three and it obeys.

That is the honest answer to give marketing: three campaigns can be done, but one of them
will be aimed at a group the data says is really two or three different kinds of customer.

### Exercise 3 — does the starting position matter?

Where k-means puts its starting centres is chosen partly at random. Run it with a different
`random_state` and see whether the answer changes.

In [ ]:
# try four different starting seeds
for seed in [0, 1, 2, 3]:
    # fit k-means with this seed
    m = KMeans(n_clusters=5, n_init=10, random_state=seed)
    # fit it and collect the group numbers
    seed_labels = m.fit_predict(X)
    # sort the group sizes so we can compare runs whose numbering differs
    sizes = np.sort(np.bincount(seed_labels))
    # print the seed and the sorted group sizes
    print("seed", seed, "-> group sizes", sizes)

**Answer.** The same five sizes every time. Three reasons: our groups are very clearly
separated, `n_init=10` means each run already tried ten different starting positions and
kept the best, and sklearn's default start deliberately spreads the centres out rather than
dropping them at random — so it rarely begins from a bad position at all.

On messier data the answer really can change from run to run. That is what `n_init` is
protecting you from — and it is why you should always set `random_state` before you show
anybody your results.

### Exercise 4 — a fake elbow

Run the elbow method on data that has **no groups at all** — just a random cloud. What does
the graph look like?

In [ ]:
# 200 points scattered completely at random, with no groups in them
noise_rng = np.random.default_rng(0)
# 200 rows of 2 columns, every number picked evenly between 0 and 100
noise = noise_rng.uniform(0, 100, size=(200, 2))

# an empty list to collect the inertia at each k
noise_inertias = []
# try the same eight values of k on the noise
for k in [1, 2, 3, 4, 5, 6, 7, 8]:
    # build a k-means with this many groups
    m = KMeans(n_clusters=k, n_init=10, random_state=0)
    # fit it to the random cloud
    m.fit(noise)
    # store the inertia
    noise_inertias.append(m.inertia_)
    # print it as we go
    print("k =", k, "  inertia =", round(m.inertia_))

In [ ]:
# draw the two elbow curves side by side
# sharey=True forces both to use the same vertical scale, so the comparison is fair
figure, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

# left: our real customers, which have a sharp corner
axes[0].plot(k_values, inertias, marker="o")
# title the left picture
axes[0].set_title("real groups: a sharp corner at 5")

# right: pure noise, which has no corner at all
axes[1].plot([1, 2, 3, 4, 5, 6, 7, 8], noise_inertias, marker="o", color="grey")
# title the right picture
axes[1].set_title("no groups: a smooth curve")

# label both
for ax in axes:
    # label the bottom axis of this picture
    ax.set_xlabel("k")
    # label the side axis of this picture
    ax.set_ylabel("inertia")

# show them
plt.show()

**Answer.** The noise curve bends **smoothly** all the way down, with no corner anywhere. It
still falls — inertia always falls — but there is no k where it suddenly stops being worth
paying for.

That is the real lesson of the elbow method: **no elbow means no groups.** k-means will
still hand you five clusters of random noise and never mention that they are meaningless.

The silhouette catches it too. Run this and compare it with Section 6.

In [ ]:
# score the noise clustering at each k, the same way we did for the real customers
for k in [2, 3, 4, 5, 6, 7, 8]:
    # build a k-means with this many groups
    m = KMeans(n_clusters=k, n_init=10, random_state=0)
    # fit it to the noise and collect the group numbers
    noise_labels = m.fit_predict(noise)
    # print the silhouette score for this k
    print("k =", k, "  silhouette on noise =", round(silhouette_score(noise, noise_labels), 4))

**Do not judge it by the level.** It is tempting to say "0.4 is low" — but our real
customers only scored 0.3736 at k = 2, which is lower than the noise manages at k = 8. A
single silhouette number, on its own, tells you very little.

**Judge it by the shape.** On the real customers the score climbed to a clear peak at 5 and
then fell away sharply. Here it just drifts between 0.37 and 0.43 with no peak at all, and
the best score is at k = 8 — the largest number we happened to try.

**Drifting, with no clear winner, is the silhouette's way of saying there is no structure
to find.**

### Exercise 5 — the summary question

In two or three sentences, explain to somebody who missed the session what clustering is
for, and what makes it harder to trust than the models we built in earlier weeks.

**Answer.** A good answer says something like: *clustering finds groups in data that has no
answer column, so it is what you reach for when nothing is labelled — customers, patients,
documents. It is harder to trust because there is no right answer to check against: you
cannot report an accuracy, the number of groups is your decision rather than the data's, and
the algorithm will happily return five groups from data that has none.*

Naming that last part — **it never refuses** — is the mark of a good answer.

---
## Recap

| | K-means | Hierarchical |
|---|---|---|
| **How it works** | pick k, place centres, nearest wins, centres move, repeat | every point starts alone, closest pair merges, repeat |
| **Choosing k** | before you start | after, by cutting the tree (with scipy) |
| **Speed** | fast on millions of rows | slow past a few thousand |
| **What you look at** | the centres | the whole dendrogram |

**Four habits worth keeping:**

1. **Scale your columns** before clustering, unless they are already in the same units.
2. Use the **elbow and the silhouette together** — and if neither shows anything, take
   seriously the possibility that there are no groups.
3. Remember that k-means always returns exactly the k you asked for, whether or not the
   groups exist.
4. Naming the groups, and deciding whether they are useful, is **your** job. The computer
   only ever says 0, 1, 2, 3, 4.